# Visual Wake Word - FP4 Model Evaluation

This notebook evaluates the original and FP4-quantized VWW models on COCO minival.

**Before running:**
1. Make sure your Google Drive has a `val2014/` folder (images) and an `annotations/` folder (with `instances_val2014.json`)
2. Update the paths in Cell 2 if your folders are in a different location

In [ ]:
#@title 1. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools opencv-python-headless
print("Done.")

In [ ]:
#@title 2. Configure Paths { display-mode: "form" }
import os

#@markdown **Google Drive paths to your COCO data:**
val2014_folder = "/content/drive/MyDrive/val2014" #@param {type:"string"}
annotations_folder = "/content/drive/MyDrive/annotations" #@param {type:"string"}

# Verify paths exist
assert os.path.isdir(val2014_folder), f"val2014 folder not found: {val2014_folder}"
assert os.path.isdir(annotations_folder), f"annotations folder not found: {annotations_folder}"

ann_file = os.path.join(annotations_folder, 'instances_val2014.json')
assert os.path.isfile(ann_file), f"instances_val2014.json not found in {annotations_folder}"

num_images = len([f for f in os.listdir(val2014_folder) if f.endswith('.jpg')])
print(f"Found {num_images} images in {val2014_folder}")
print(f"Found annotations at {ann_file}")
print("All good!")

In [ ]:
#@title 3. Create symlinked data directory & clone repo
import os

# Create the directory structure eval_fp4_coco.py expects
os.makedirs('/content/coco/raw-data', exist_ok=True)

# Symlink to Drive folders (no copying needed!)
if not os.path.exists('/content/coco/raw-data/val2014'):
    os.symlink(val2014_folder, '/content/coco/raw-data/val2014')
if not os.path.exists('/content/coco/raw-data/annotations'):
    os.symlink(annotations_folder, '/content/coco/raw-data/annotations')

print("Symlinks created:")
!ls -la /content/coco/raw-data/

# Clone repo
!git clone https://github.com/amitmate/visualwakeword.git /content/vww 2>/dev/null || (cd /content/vww && git pull)

# Copy minival IDs
import shutil
shutil.copy('/content/vww/mscocominival.txt', '/content/coco/mscocominival.txt')
print("\nRepo cloned. Ready to evaluate!")

In [ ]:
#@title 4. Run Evaluation
%cd /content/vww
!python eval_fp4_coco.py \
    --data-dir /content/coco/raw-data \
    --minival-ids /content/coco/mscocominival.txt \
    --h5-model modelVisualWakeWord.h5 \
    --tflite-model modelVisualWakeWord.tflite \
    --fp4-bin compressed_models/model_mxfp4.fp4bin \
    --fp4-tflite compressed_models/model_mxfp4_int8.tflite